In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score
)

RS = 42
PRESUPUESTO = 2_000_000

In [2]:
train = pd.read_csv("solicitudes_train.csv")
test = pd.read_csv("solicitudes_test.csv")
banco = pd.read_csv("comportamiento_bancario.csv")

In [3]:
print("Solicitudes de entrenamiento:", train.shape)
print("Solicitudes de prueba:", test.shape)
print("Historial bancario:", banco.shape)

Solicitudes de entrenamiento: (32416, 8)
Solicitudes de prueba: (3061, 7)
Historial bancario: (661139, 4)


In [4]:
train.head()

,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Fecha_Solicitud,Estado_Final
0,1,12698,7900,0.2333,4461,Consumo,2025-04,1
1,2,20744,17400,0.2507,3889,Vivienda,2025-12,1
2,3,6830,8600,0.2368,4035,Vehiculo,2025-06,0
3,4,5722,13300,0.2826,4018,Vehiculo,2025-07,1
4,5,24719,6600,0.2314,3357,Otros,2025-01,1


In [5]:
test.head()

,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Fecha_Solicitud
0,32417,27074,2300,0.2771,2596,Consumo,2026-01
1,32418,28172,10900,0.3291,6905,Consumo,2026-01
2,32419,27547,1300,0.2485,3707,Educacion,2026-01
3,32420,27689,1600,0.2699,3314,Consumo,2026-01
4,32421,26219,900,0.2317,3870,Educacion,2026-01


In [6]:
banco.head()

,ID_Usuario,Mes_Referencia,Saldo_Promedio_Mensual,Dias_Atraso_Otras_Deudas
0,1,2024-01,12768.11,3
1,1,2024-02,15817.67,3
2,1,2024-03,13138.66,0
3,1,2024-04,19453.13,0
4,1,2024-05,14943.44,59


In [7]:
test.dtypes

ID_Solicitud          int64
ID_Usuario            int64
Monto_Solicitado      int64
Tasa_Interes        float64
Ingreso_Mensual       int64
Motivo_Prestamo      object
Fecha_Solicitud      object
dtype: object

In [8]:
train.dtypes

ID_Solicitud          int64
ID_Usuario            int64
Monto_Solicitado      int64
Tasa_Interes        float64
Ingreso_Mensual       int64
Motivo_Prestamo      object
Fecha_Solicitud      object
Estado_Final          int64
dtype: object

In [9]:
banco.dtypes

ID_Usuario                    int64
Mes_Referencia               object
Saldo_Promedio_Mensual      float64
Dias_Atraso_Otras_Deudas      int64
dtype: object

In [10]:
train.isnull().sum()

ID_Solicitud        0
ID_Usuario          0
Monto_Solicitado    0
Tasa_Interes        0
Ingreso_Mensual     0
Motivo_Prestamo     0
Fecha_Solicitud     0
Estado_Final        0
dtype: int64

In [11]:
test.isnull().sum()

ID_Solicitud        0
ID_Usuario          0
Monto_Solicitado    0
Tasa_Interes        0
Ingreso_Mensual     0
Motivo_Prestamo     0
Fecha_Solicitud     0
dtype: int64

In [12]:
banco.isnull().sum()

ID_Usuario                  0
Mes_Referencia              0
Saldo_Promedio_Mensual      0
Dias_Atraso_Otras_Deudas    0
dtype: int64

In [13]:
train.duplicated().sum()

0

In [14]:
test.duplicated().sum()

0

In [15]:
banco.duplicated().sum()

0

In [16]:
train["Fecha_Solicitud"] = pd.to_datetime(
    train["Fecha_Solicitud"], format="%Y-%m"
)
test["Fecha_Solicitud"] = pd.to_datetime(
    test["Fecha_Solicitud"], format="%Y-%m"
)

banco["Mes_Referencia"] = pd.to_datetime(
    banco["Mes_Referencia"], format="%Y-%m"
)
print(train["Fecha_Solicitud"].head())
print(test["Fecha_Solicitud"].head())
print(banco["Mes_Referencia"].head())

0   2025-04-01
1   2025-12-01
2   2025-06-01
3   2025-07-01
4   2025-01-01
Name: Fecha_Solicitud, dtype: datetime64[ns]
0   2026-01-01
1   2026-01-01
2   2026-01-01
3   2026-01-01
4   2026-01-01
Name: Fecha_Solicitud, dtype: datetime64[ns]
0   2024-01-01
1   2024-02-01
2   2024-03-01
3   2024-04-01
4   2024-05-01
Name: Mes_Referencia, dtype: datetime64[ns]


In [17]:
print("Filas de banco:", len(banco))
print("Usuarios diferentes:", banco["ID_Usuario"].nunique())
print("Solicitudes de entrenamiento:", len(train))
print("Solicitudes de prueba:", len(test))

Filas de banco: 661139
Usuarios diferentes: 28993
Solicitudes de entrenamiento: 32416
Solicitudes de prueba: 3061


In [18]:
def agregar_historial(solicitudes, banco):
    resultado = solicitudes.copy()
    agregados = []
    for i in range(len(resultado)):
        id_solicitud = resultado["ID_Solicitud"].iloc[i]
        id_usuario = resultado["ID_Usuario"].iloc[i]
        fecha = resultado["Fecha_Solicitud"].iloc[i]

        historial = banco[
            (banco["ID_Usuario"] == id_usuario) &
            (banco["Mes_Referencia"] < fecha)
        ]

        historial = historial.sort_values("Mes_Referencia")

        if len(historial) == 0:
            saldo_promedio = 0
            saldo_minimo = 0
            saldo_maximo = 0
            atraso_promedio = 0
            atraso_maximo = 0
            meses_historial = 0

        else:
            saldo_promedio = historial[
                "Saldo_Promedio_Mensual"
            ].mean()

            saldo_minimo = historial[
                "Saldo_Promedio_Mensual"
            ].min()

            saldo_maximo = historial[
                "Saldo_Promedio_Mensual"
            ].max()

            atraso_promedio = historial[
                "Dias_Atraso_Otras_Deudas"
            ].mean()

            atraso_maximo = historial[
                "Dias_Atraso_Otras_Deudas"
            ].max()

            meses_historial = len(historial)

        agregados.append({
            "ID_Solicitud": id_solicitud,
            "saldo_promedio": saldo_promedio,
            "saldo_minimo": saldo_minimo,
            "saldo_maximo": saldo_maximo,
            "atraso_promedio": atraso_promedio,
            "atraso_maximo": atraso_maximo,
            "meses_historial": meses_historial
        })

    agregados = pd.DataFrame(agregados)
    agregados = agregados.set_index("ID_Solicitud")

    resultado = resultado.join(
        agregados,
        on="ID_Solicitud",
        how="left"
    )
    return resultado

In [19]:
prueba = agregar_historial(train.head(), banco)
print(prueba[
    [
        "ID_Solicitud",
        "saldo_promedio",
        "saldo_minimo",
        "saldo_maximo",
        "atraso_promedio",
        "atraso_maximo",
        "meses_historial"
    ]
])

   ID_Solicitud  saldo_promedio  saldo_minimo  saldo_maximo  atraso_promedio  \
0             1     3695.225714       3121.25       4698.37         4.857143   
1             2     5976.456667       3807.48       7011.74         8.809524   
2             3      178.574667        154.88        209.05        20.733333   
3             4     7589.356667       5949.24       9501.68         0.944444   
4             5     3888.616667       2952.59       4848.48         3.750000   

   atraso_maximo  meses_historial  
0             25               14  
1             56               21  
2             57               15  
3              4               18  
4             32               12  


In [20]:
train_preparado = agregar_historial(train, banco)
test_preparado = agregar_historial(test, banco)

print("Entrenamiento preparado:", train_preparado.shape)
print("Prueba preparada:", test_preparado.shape)

print(train_preparado.head())
print("\nValores nulos:")
print(train_preparado[
    [
        "saldo_promedio",
        "saldo_minimo",
        "saldo_maximo",
        "atraso_promedio",
        "atraso_maximo",
        "meses_historial"
    ]
].isnull().sum())

Entrenamiento preparado: (32416, 14)
Prueba preparada: (3061, 13)
   ID_Solicitud  ID_Usuario  Monto_Solicitado  Tasa_Interes  Ingreso_Mensual  \
0             1       12698              7900        0.2333             4461   
1             2       20744             17400        0.2507             3889   
2             3        6830              8600        0.2368             4035   
3             4        5722             13300        0.2826             4018   
4             5       24719              6600        0.2314             3357   

  Motivo_Prestamo Fecha_Solicitud  Estado_Final  saldo_promedio  saldo_minimo  \
0         Consumo      2025-04-01             1     3695.225714       3121.25   
1        Vivienda      2025-12-01             1     5976.456667       3807.48   
2        Vehiculo      2025-06-01             0      178.574667        154.88   
3        Vehiculo      2025-07-01             1     7589.356667       5949.24   
4           Otros      2025-01-01             1 

In [21]:
entrenamiento = train_preparado[
    train_preparado["Fecha_Solicitud"] < "2025-11-01"
].copy()

validacion = train_preparado[
    train_preparado["Fecha_Solicitud"] >= "2025-11-01"
].copy()

print("Solicitudes para entrenar:", len(entrenamiento))
print("Solicitudes para validar:", len(validacion))

print("\nMeses de entrenamiento:")
print(entrenamiento["Fecha_Solicitud"].dt.month.value_counts().sort_index())

print("\nMeses de validación:")
print(validacion["Fecha_Solicitud"].dt.month.value_counts().sort_index())

Solicitudes para entrenar: 27106
Solicitudes para validar: 5310

Meses de entrenamiento:
Fecha_Solicitud
1     2628
2     2742
3     2745
4     2675
5     2702
6     2730
7     2709
8     2751
9     2709
10    2715
Name: count, dtype: int64

Meses de validación:
Fecha_Solicitud
11    2703
12    2607
Name: count, dtype: int64


In [22]:
columnas_numericas = [
    "Monto_Solicitado",
    "Tasa_Interes",
    "Ingreso_Mensual",
    "saldo_promedio",
    "saldo_minimo",
    "saldo_maximo",
    "atraso_promedio",
    "atraso_maximo",
    "meses_historial"
]

columnas_categoricas = [
    "Motivo_Prestamo"
]

columnas_modelo = columnas_numericas + columnas_categoricas

X_train = entrenamiento[columnas_modelo]
y_train = entrenamiento["Estado_Final"]

X_val = validacion[columnas_modelo]
y_val = validacion["Estado_Final"]

print("Variables predictoras:", columnas_modelo)
print("Cantidad de variables:", len(columnas_modelo))
print("Filas para entrenar:", len(X_train))
print("Filas para validar:", len(X_val))

Variables predictoras: ['Monto_Solicitado', 'Tasa_Interes', 'Ingreso_Mensual', 'saldo_promedio', 'saldo_minimo', 'saldo_maximo', 'atraso_promedio', 'atraso_maximo', 'meses_historial', 'Motivo_Prestamo']
Cantidad de variables: 10
Filas para entrenar: 27106
Filas para validar: 5310


In [23]:
preprocesamiento = ColumnTransformer([
    (
        "numericas",
        SimpleImputer(strategy="median"),
        columnas_numericas
    ),
    (
        "categoricas",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        columnas_categoricas
    )
])

print("Preprocesamiento preparado")

Preprocesamiento preparado


In [24]:
modelo_rf = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("modelo", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=20,
        max_features=0.8,
        class_weight="balanced",
        random_state=RS,
        n_jobs=-1
    ))
])

modelo_rf.fit(X_train, y_train)

print("Modelo entrenado")

Modelo entrenado


In [25]:
prob_train = modelo_rf.predict_proba(X_train)[:, 1]
prob_val = modelo_rf.predict_proba(X_val)[:, 1]

auc_train = roc_auc_score(y_train, prob_train)
auc_val = roc_auc_score(y_val, prob_val)

brecha = auc_train - auc_val

print("AUC entrenamiento:", (auc_train))
print("AUC validación:", (auc_val))
print("Brecha entre ambos:", (brecha))

AUC entrenamiento: 0.901789231984141
AUC validación: 0.8400784942147863
Brecha entre ambos: 0.06171073776935465


In [26]:
resultados = validacion.copy()
resultados["Probabilidad_Pago"] = prob_val

resultados["Ganancia_Esperada"] = (
    resultados["Monto_Solicitado"] *
    (
        resultados["Probabilidad_Pago"] *
        (1 + resultados["Tasa_Interes"]) - 1
    )
)

resultados["Rendimiento"] = (
    resultados["Ganancia_Esperada"] /
    resultados["Monto_Solicitado"]
)

candidatos = resultados[
    resultados["Ganancia_Esperada"] > 0
].sort_values("Rendimiento", ascending=False)

capital_usado = 0
indices_aprobados = []

for i in range(len(candidatos)):
    fila = candidatos.iloc[i]
    indice = candidatos.index[i]
    monto = fila["Monto_Solicitado"]

    if capital_usado + monto <= PRESUPUESTO:
        indices_aprobados.append(indice)
        capital_usado = capital_usado + monto

aprobados = resultados.loc[indices_aprobados]

ganancia_real = 0

for i in range(len(aprobados)):
    monto = aprobados["Monto_Solicitado"].iloc[i]
    tasa = aprobados["Tasa_Interes"].iloc[i]
    estado = aprobados["Estado_Final"].iloc[i]

    if estado == 1:
        ganancia_real = ganancia_real + monto * tasa
    else:
        ganancia_real = ganancia_real - monto

print("Préstamos aprobados:", len(aprobados))
print("Capital utilizado: $", (capital_usado))
print("Presupuesto restante: $", (PRESUPUESTO - capital_usado))
print("Ganancia real: $", (ganancia_real))
print("ROI:", (ganancia_real / capital_usado))

Préstamos aprobados: 149
Capital utilizado: $ 1999500
Presupuesto restante: $ 500
Ganancia real: $ 495485.83000000013
ROI: 0.2478048662165542


In [27]:
modelo_rf_15 = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("modelo", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=15,
        max_features=0.8,
        class_weight="balanced",
        random_state=RS,
        n_jobs=-1
    ))
])

modelo_rf_15.fit(X_train, y_train)

prob_train_15 = modelo_rf_15.predict_proba(X_train)[:, 1]
prob_val_15 = modelo_rf_15.predict_proba(X_val)[:, 1]

auc_train_15 = roc_auc_score(y_train, prob_train_15)
auc_val_15 = roc_auc_score(y_val, prob_val_15)

print("AUC entrenamiento:", (auc_train_15))
print("AUC validación:", (auc_val_15))
print("Brecha:", (auc_train_15 - auc_val_15))

AUC entrenamiento: 0.9070367192705203
AUC validación: 0.8409158563745993
Brecha: 0.066120862895921


In [28]:
resultados_15 = validacion.copy()

resultados_15["Probabilidad_Pago"] = prob_val_15

resultados_15["Ganancia_Esperada"] = (
    resultados_15["Monto_Solicitado"] *
    (
        resultados_15["Probabilidad_Pago"] *
        (1 + resultados_15["Tasa_Interes"]) - 1
    )
)

resultados_15["Rendimiento"] = (
    resultados_15["Ganancia_Esperada"] /
    resultados_15["Monto_Solicitado"]
)

candidatos_15 = resultados_15[
    resultados_15["Ganancia_Esperada"] > 0
].sort_values("Rendimiento", ascending=False)

capital_usado_15 = 0
indices_aprobados_15 = []

for i in range(len(candidatos_15)):
    fila = candidatos_15.iloc[i]
    indice = candidatos_15.index[i]
    monto = fila["Monto_Solicitado"]

    if capital_usado_15 + monto <= PRESUPUESTO:
        indices_aprobados_15.append(indice)
        capital_usado_15 = capital_usado_15 + monto

aprobados_15 = resultados_15.loc[indices_aprobados_15]

ganancia_real_15 = 0

for i in range(len(aprobados_15)):
    monto = aprobados_15["Monto_Solicitado"].iloc[i]
    tasa = aprobados_15["Tasa_Interes"].iloc[i]
    estado = aprobados_15["Estado_Final"].iloc[i]

    if estado == 1:
        ganancia_real_15 = ganancia_real_15 + monto * tasa
    else:
        ganancia_real_15 = ganancia_real_15 - monto

print("Préstamos aprobados:", len(aprobados_15))
print("Capital utilizado: $", (capital_usado_15))
print("Presupuesto restante: $", (PRESUPUESTO - capital_usado_15))
print("Ganancia real: $", (ganancia_real_15))
print("ROI:", (ganancia_real_15 / capital_usado_15))

Préstamos aprobados: 167
Capital utilizado: $ 1999900
Presupuesto restante: $ 100
Ganancia real: $ 515777.7500000001
ROI: 0.25790177008850446


In [29]:
modelo_rf_10 = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("modelo", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=10,
        max_features=0.8,
        class_weight="balanced",
        random_state=RS,
        n_jobs=-1
    ))
])

modelo_rf_10.fit(X_train, y_train)

prob_train_10 = modelo_rf_10.predict_proba(X_train)[:, 1]
prob_val_10 = modelo_rf_10.predict_proba(X_val)[:, 1]

auc_train_10 = roc_auc_score(y_train, prob_train_10)
auc_val_10 = roc_auc_score(y_val, prob_val_10)

print("AUC entrenamiento:", (auc_train_10))
print("AUC validación:", (auc_val_10))
print("Brecha:", (auc_train_10 - auc_val_10))

AUC entrenamiento: 0.9148598323926675
AUC validación: 0.8400372100602965
Brecha: 0.07482262233237102


In [30]:
resultados_10 = validacion.copy()
resultados_10["Probabilidad_Pago"] = prob_val_10

resultados_10["Ganancia_Esperada"] = (
    resultados_10["Monto_Solicitado"] *
    (
        resultados_10["Probabilidad_Pago"] *
        (1 + resultados_10["Tasa_Interes"]) - 1
    )
)

resultados_10["Rendimiento"] = (
    resultados_10["Ganancia_Esperada"] /
    resultados_10["Monto_Solicitado"]
)

candidatos_10 = resultados_10[
    resultados_10["Ganancia_Esperada"] > 0
].sort_values("Rendimiento", ascending=False)

capital_usado_10 = 0
indices_aprobados_10 = []

for i in range(len(candidatos_10)):
    fila = candidatos_10.iloc[i]
    indice = candidatos_10.index[i]
    monto = fila["Monto_Solicitado"]

    if capital_usado_10 + monto <= PRESUPUESTO:
        indices_aprobados_10.append(indice)
        capital_usado_10 = capital_usado_10 + monto

aprobados_10 = resultados_10.loc[indices_aprobados_10]

ganancia_real_10 = 0

for i in range(len(aprobados_10)):
    monto = aprobados_10["Monto_Solicitado"].iloc[i]
    tasa = aprobados_10["Tasa_Interes"].iloc[i]
    estado = aprobados_10["Estado_Final"].iloc[i]

    if estado == 1:
        ganancia_real_10 = ganancia_real_10 + monto * tasa
    else:
        ganancia_real_10 = ganancia_real_10 - monto

print("Préstamos aprobados:", len(aprobados_10))
print("Capital utilizado: $", (capital_usado_10))
print("Ganancia real: $", (ganancia_real_10))
print("ROI:", (ganancia_real_10 / capital_usado_10))

Préstamos aprobados: 145
Capital utilizado: $ 1999800
Ganancia real: $ 525130.6200000001
ROI: 0.2625915691569157


In [31]:
X_total = train_preparado[columnas_modelo]
y_total = train_preparado["Estado_Final"]

modelo_final = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("modelo", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=10,
        max_features=0.8,
        class_weight="balanced",
        random_state=RS,
        n_jobs=-1
    ))
])

modelo_final.fit(X_total, y_total)

print("Modelo final entrenado.")

Modelo final entrenado.


In [32]:
X_test = test_preparado[columnas_modelo]
prob_test = modelo_final.predict_proba(X_test)[:, 1]

resultados_test = test_preparado.copy()
resultados_test["Probabilidad_Pago"] = prob_test

resultados_test["Ganancia_Esperada"] = (
    resultados_test["Monto_Solicitado"] *
    (
        resultados_test["Probabilidad_Pago"] *
        (1 + resultados_test["Tasa_Interes"]) - 1
    )
)

resultados_test["Rendimiento"] = (
    resultados_test["Ganancia_Esperada"] /
    resultados_test["Monto_Solicitado"]
)

candidatos_test = resultados_test[
    resultados_test["Ganancia_Esperada"] > 0
].sort_values("Rendimiento", ascending=False)

capital_usado = 0
indices_aprobados = []

for i in range(len(candidatos_test)):
    fila = candidatos_test.iloc[i]
    indice = candidatos_test.index[i]
    monto = fila["Monto_Solicitado"]

    if capital_usado + monto <= PRESUPUESTO:
        indices_aprobados.append(indice)
        capital_usado = capital_usado + monto

resultados_test["Decision_Aprobacion"] = 0
resultados_test.loc[indices_aprobados, "Decision_Aprobacion"] = 1

print("Préstamos aprobados:", len(indices_aprobados))
print("Capital utilizado: $", (capital_usado))
print("Presupuesto restante: $", (PRESUPUESTO - capital_usado))

Préstamos aprobados: 743
Capital utilizado: $ 2000000
Presupuesto restante: $ 0


In [36]:
entrenamiento_nuevo = train_preparado[
    train_preparado["Fecha_Solicitud"] < "2025-09-01"
].copy()

validacion_nueva = train_preparado[
    (train_preparado["Fecha_Solicitud"] >= "2025-09-01") &
    (train_preparado["Fecha_Solicitud"] < "2025-11-01")
].copy()

X_train_nuevo = entrenamiento_nuevo[columnas_modelo]
y_train_nuevo = entrenamiento_nuevo["Estado_Final"]

X_val_nuevo = validacion_nueva[columnas_modelo]
y_val_nuevo = validacion_nueva["Estado_Final"]

print("Solicitudes para entrenar:", len(X_train_nuevo))
print("Solicitudes para validar:", len(X_val_nuevo))
print(
    "Meses de validación:",
    validacion_nueva["Fecha_Solicitud"].dt.month.unique()
)

Solicitudes para entrenar: 21682
Solicitudes para validar: 5424
Meses de validación: [ 9 10]


In [37]:
modelo_prueba = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("modelo", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=10,
        max_features=0.8,
        class_weight="balanced",
        random_state=RS,
        n_jobs=-1
    ))
])

modelo_prueba.fit(X_train_nuevo, y_train_nuevo)

prob_train_nuevo = modelo_prueba.predict_proba(
    X_train_nuevo
)[:, 1]

prob_val_nuevo = modelo_prueba.predict_proba(
    X_val_nuevo
)[:, 1]

auc_train_nuevo = roc_auc_score(
    y_train_nuevo, prob_train_nuevo
)

auc_val_nuevo = roc_auc_score(
    y_val_nuevo, prob_val_nuevo
)

print("AUC entrenamiento:", (auc_train_nuevo))
print("AUC validación:", (auc_val_nuevo))
print(
    "Brecha:",
    (auc_train_nuevo - auc_val_nuevo)
)

AUC entrenamiento: 0.9192035616449368
AUC validación: 0.8466574784762833
Brecha: 0.07254608316865352


In [38]:
resultados_nuevos = validacion_nueva.copy()
resultados_nuevos["Probabilidad_Pago"] = prob_val_nuevo

resultados_nuevos["Ganancia_Esperada"] = (
    resultados_nuevos["Monto_Solicitado"] *
    (
        resultados_nuevos["Probabilidad_Pago"] *
        (1 + resultados_nuevos["Tasa_Interes"]) - 1
    )
)

resultados_nuevos["Rendimiento"] = (
    resultados_nuevos["Ganancia_Esperada"] /
    resultados_nuevos["Monto_Solicitado"]
)

candidatos_nuevos = resultados_nuevos[
    resultados_nuevos["Ganancia_Esperada"] > 0
].sort_values("Rendimiento", ascending=False)

capital_nuevo = 0
indices_nuevos = []

for i in range(len(candidatos_nuevos)):
    fila = candidatos_nuevos.iloc[i]
    indice = candidatos_nuevos.index[i]
    monto = fila["Monto_Solicitado"]

    if capital_nuevo + monto <= PRESUPUESTO:
        indices_nuevos.append(indice)
        capital_nuevo = capital_nuevo + monto

aprobados_nuevos = resultados_nuevos.loc[indices_nuevos]

ganancia_nueva = 0

for i in range(len(aprobados_nuevos)):
    monto = aprobados_nuevos["Monto_Solicitado"].iloc[i]
    tasa = aprobados_nuevos["Tasa_Interes"].iloc[i]
    estado = aprobados_nuevos["Estado_Final"].iloc[i]

    if estado == 1:
        ganancia_nueva = ganancia_nueva + monto * tasa
    else:
        ganancia_nueva = ganancia_nueva - monto

print("Préstamos aprobados:", len(aprobados_nuevos))
print("Capital utilizado: $", (capital_nuevo))
print("Ganancia real: $", (ganancia_nueva))

if capital_nuevo > 0:
    print("ROI:", (ganancia_nueva / capital_nuevo))

Préstamos aprobados: 132
Capital utilizado: $ 1999900
Ganancia real: $ 550103.91
ROI: 0.27506570828541427


In [39]:
prediccion_final = resultados_test[
    ["ID_Solicitud", "Decision_Aprobacion"]
].copy()

prediccion_final.to_csv("PrediccionRF2.csv", index=False)

print("Archivo generado: PrediccionRF2.csv")
print("Filas:", len(prediccion_final))
print("Columnas:", list(prediccion_final.columns))

Archivo generado: PrediccionRF2.csv
Filas: 3061
Columnas: ['ID_Solicitud', 'Decision_Aprobacion']


In [41]:
from IPython.display import FileLink, display
display(FileLink("PrediccionRF2.csv"))

/home/trinidadbonzi/machine-learning/prestamatch/PrediccionRF2.csv